# omicsTL - Transfer Learning Example

<div>
<img src="images/tl_arch1.png" width="500" style="background-color: white;"/>
</div>

This notebook demonstrates a complete transfer-learning workflow using omicsTL. For clarity and reproducibility, we generate linked synthetic source and target datasets from two real viral proteomics datasets. This synthetic-data step is used here to showcase omicsTL’s simulation utilities and to create controlled source/target domain differences.

In practice, you can skip the simulation step and apply the same modeling workflow directly to real source and target datasets with minimal changes.

## Workflow summary

1. Load two real datasets (source and target).

2. Generate linked synthetic datasets with either a continuous or categorical response (for demonstration purposes only-not needed for real data application).

3. Split the synthetic target dataset into train, ensemble, and test partitions.

4. Package everything into a DatasetContainer.

5. Fit transfer learning models (deep learning and random forest variants).

6. Compare transfer learning vs target-only baselines.

# Imports and core utilities
We start by importing the simulation utilities needed to define a response function and generate synthetic data.

In [1]:
import warnings
warnings.filterwarnings("always")

from omicstl.simulation_utils.data_generation import response_function, generate_synth_data

# Load real datasets and define simulation settings
We use two real datasets with aligned features. The first represents the source domain used to train the base model; the second represents the target domain used for adaptation and evaluation.

We also define a non-trivial response function and choose sample sizes and the number of generated features.

In [2]:
import pandas as pd
import numpy as np

source_real_data_path = "data/source_data_real.csv"
target_real_data_path = "data/target_data_real.csv"

source_real_data = pd.read_csv(source_real_data_path, index_col=0)
target_real_data = pd.read_csv(target_real_data_path, index_col=0)

# Non-trivial continuous response function
response_fn = response_function("tanh(df[, 2]) + df[, 1] * df[, ncol(df)] ^ 2")

num_features = 100
num_samples_source = 100
num_samples_target = 50
samples_target_test = 25
samples_target_ensemble = 5

 # Generate linked synthetic datasets (continuous and categorical)

 We generate linked synthetic datasets for both a continuous and categorical response. The critical piece is passing *`prior_lc_info`* from the source simulation into the target simulation. This ensures the source and target synthetic datasets are coupled and reflect a coherent transfer-learning scenario

In [3]:
# Continuous response

source_synth_data_cont, source_lc_info_cont, _ = generate_synth_data(
    data = source_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_source, # number of output samples
    response_fn = response_fn, # response function
    snr = 1 # signal to noise ratio
)

target_synth_data_cont, _, _ = generate_synth_data(
    data = target_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_target + samples_target_test + samples_target_ensemble, # number of output samples
    response_fn = response_fn, # response function
    prior_lc_info = source_lc_info_cont, # crucial to include else the source and target datasets aren't linked!
    snr = 1 # signal to noise ratio
)

display(source_synth_data_cont)
display(target_synth_data_cont)

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-0.018577,-2.131510,4.221440,7.503035,0.704963,4.994967,1.974920,-1.389276,-2.121761,-2.686822,...,5.012847,1.029205,-1.485498,3.928012,0.794997,0.670302,2.786066,8.491732,-2.031756,0.116399
1,1.647239,-2.541027,4.464055,8.170037,0.448550,5.254451,1.639822,-1.552928,-2.047237,-2.977847,...,5.153622,0.895523,-1.169409,3.986608,0.025046,0.251345,2.805506,9.045304,-2.157403,-0.023953
2,-1.212490,-2.140310,4.584173,8.010408,0.552976,5.319696,2.019203,-1.104839,-3.188980,-2.853624,...,3.255986,0.781753,-1.217508,4.504326,0.987089,0.886421,2.750850,8.694841,-2.431166,-0.693216
3,1.717835,-2.476493,4.213687,7.685149,0.655794,4.797119,2.243716,-1.213976,-0.911602,-3.179669,...,5.890727,0.987917,-1.346762,4.099814,1.204830,0.611966,2.748806,8.128385,-1.751405,0.239025
4,0.828215,-2.559346,4.489350,7.722889,0.816977,4.869013,1.448423,-1.378270,-1.312596,-2.537707,...,6.329980,0.814059,-1.101288,4.431397,0.487904,0.434994,2.410800,8.530311,-1.780194,0.533092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1.132352,-2.575378,4.335910,7.880893,0.511562,4.968101,1.666043,-1.319583,-3.059473,-3.048925,...,4.186401,0.904709,-1.366982,4.172993,1.185925,0.224145,2.943122,8.613989,-2.507009,0.130276
96,0.416058,-2.492329,4.535006,8.181652,0.722679,4.787186,1.217221,-1.232186,-2.207171,-3.422057,...,3.492457,0.878575,-1.523754,4.149942,1.440319,0.675295,2.510847,8.433688,-2.334204,0.418409
97,0.055745,-1.790082,4.298029,7.858824,0.787478,5.324826,2.256585,-1.766166,-0.285702,-3.021066,...,4.941716,0.967434,-1.516619,4.438812,0.084988,0.480843,1.962870,8.318415,-2.064034,0.054013
98,1.435158,-2.712868,3.964359,7.665058,0.461145,5.245588,1.677615,-1.117710,-0.714720,-2.777461,...,3.261225,0.617396,-1.425893,4.535557,0.698371,0.579830,2.726510,8.354184,-2.074454,0.094469


,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-0.131928,-1.825388,4.387839,8.065345,0.542474,5.000099,1.189423,-1.161513,-0.968273,-2.778407,...,2.322157,0.957780,-1.387260,4.247323,1.862496,0.799520,2.320422,8.470030,-1.849864,0.006803
1,0.145121,-2.554434,4.349408,7.799595,0.387225,5.320900,1.349798,-1.331643,-2.065110,-2.948631,...,3.990292,0.845537,-1.172766,4.262803,1.541075,0.495177,1.948710,8.750320,-1.721811,-0.247589
2,1.622457,-1.943416,4.287696,7.768383,0.370545,5.013423,1.833743,-0.925606,-1.942006,-2.827866,...,3.097417,0.698807,-1.799050,4.567008,0.669068,0.823613,2.333797,8.505746,-2.298207,0.077588
3,1.362953,-1.891210,4.447942,7.572323,0.720992,5.378908,2.000493,-0.968341,-2.112151,-2.904019,...,1.405052,0.890627,-1.695162,3.967584,0.883925,0.932713,2.304135,8.507120,-2.367570,-0.502198
4,1.337630,-2.151685,4.389698,8.134839,0.547072,4.738690,1.571785,-1.034105,-1.566045,-3.522164,...,5.730517,0.954667,-1.422893,4.314065,0.322956,0.751003,2.692535,8.746863,-2.003059,-0.131694
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,-3.238152,-2.887291,4.349457,7.760678,0.592905,4.904736,1.209448,-1.349284,0.546408,-3.096531,...,4.549129,1.065473,-1.395109,4.207340,1.866048,0.395522,2.346771,8.545214,-2.304675,-1.011131
76,1.249571,-2.329088,4.125388,8.215697,0.428899,5.388900,2.020143,-1.191649,-2.372020,-2.690694,...,4.036303,0.840662,-1.407928,3.793202,1.107406,0.622723,2.509919,8.426898,-2.156630,-0.015322
77,0.284994,-2.403273,4.254410,7.832009,0.831893,5.040045,2.132998,-0.949456,-0.619289,-3.058549,...,4.655577,0.843374,-1.447009,4.385909,1.809769,0.916229,2.527795,8.567823,-2.141750,-0.628910
78,-0.294250,-2.271097,4.178750,8.105634,0.627029,5.154989,2.121331,-1.058402,0.449825,-3.090483,...,2.707273,0.884918,-1.787573,4.444146,2.515218,0.306426,2.789173,8.509702,-2.612548,-0.807883


In [ ]:
# Categorical response
source_synth_data_cat, source_lc_info_cat, _ = generate_synth_data(
    data = source_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_source + samples_target_ensemble, # number of output samples
    response_fn = response_fn, # response function
    snr = 1, # signal to noise ratio
    response_parameters={
        "ncats": 3,
        "quantile": "quantile"
    }
)

target_synth_data_cat, source_lc_info_cat, _ = generate_synth_data(
    data = target_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_target + samples_target_test + samples_target_ensemble, # number of output samples
    response_fn = response_fn, # response function
    snr = 1 ,# signal to noise ratio
    prior_lc_info = source_lc_info_cat, # crucial to include else the source and target datasets aren't linked!
    response_parameters={
        "ncats": 3,
        "quantile": "quantile"
    }
)

display(source_synth_data_cat)
display(target_synth_data_cat)

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,2.0,3.184250,3.445221,0.772872,1.583466,0.188274,1.153636,7.081006,1.970003,-1.144194,...,-2.264175,-0.338430,3.798686,-0.374314,-0.987292,7.880432,1.669029,2.783176,-0.344493,5.073896
1,1.0,2.930739,3.444411,0.565156,0.635906,0.603209,0.838928,7.949908,1.815713,-1.043961,...,-1.873429,-0.658501,4.181844,-1.009165,-1.530263,7.770458,1.524477,1.700582,-0.206422,4.744504
2,3.0,4.061349,3.515256,0.491728,0.941261,0.290420,1.069911,7.539802,2.173966,-1.421109,...,-2.295036,-1.128794,3.475350,-0.672629,-0.816676,8.028649,2.290329,2.503215,-0.620180,4.722170
3,2.0,3.791040,3.619319,0.509547,0.460347,0.197607,0.954524,7.544457,1.743920,-1.260519,...,-2.001552,-0.127618,3.659228,-0.825454,-1.261197,7.912557,1.713280,2.226306,-0.618549,4.681098
4,1.0,2.324803,3.404648,0.488172,1.238389,0.618491,0.491636,7.559692,2.095115,-1.114257,...,-2.764871,-0.591379,3.512712,-1.252520,-1.655410,8.000330,2.194835,2.262948,-0.740519,5.113681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,1.0,3.391414,3.681952,0.414090,1.318297,0.626547,0.641115,7.406402,2.288608,-1.440167,...,-2.065471,-0.570546,3.363025,-0.776700,-0.606547,7.973992,2.158482,2.476712,-0.548647,4.712099
101,3.0,3.921791,3.649770,0.420173,0.894762,0.247528,0.707007,7.203413,2.131378,-0.779443,...,-2.076712,-0.732730,3.685996,-1.238457,-0.856233,7.811980,1.985250,2.263119,-0.263122,4.949613
102,2.0,4.233251,3.791196,0.597181,1.173061,-0.034478,0.720862,7.619725,2.062568,-1.043749,...,-2.150644,0.097892,3.276727,-0.888212,-1.241449,7.801340,2.124649,2.132417,0.195482,4.766642
103,2.0,3.228267,3.563340,0.648522,0.575898,0.073183,1.004983,7.865742,1.855545,-0.940535,...,-2.700999,-0.883108,3.851675,-0.359209,-0.950857,7.855413,2.192433,2.671906,-1.089740,4.788992


,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,3.0,3.587775,3.750322,0.607150,0.011245,0.036504,1.147931,7.326725,1.589885,-1.268487,...,-1.654367,-0.911683,3.811610,-0.667263,-1.172213,7.895552,1.401248,2.664864,-0.891760,5.009576
1,1.0,2.162534,3.402201,0.582610,1.142536,0.045415,1.054657,6.835663,1.871839,-0.982427,...,-2.118626,-0.711217,3.706629,-1.181281,-1.335658,7.990837,1.993115,2.819070,-0.854748,4.880440
2,1.0,2.620521,3.395528,0.395628,0.508721,0.264623,0.616177,7.619991,1.785727,-1.117486,...,-2.033089,-0.655606,3.628446,0.146929,-1.233156,7.879871,1.416449,1.545361,0.119349,4.977223
3,2.0,2.339685,3.789641,0.195729,0.308257,-0.063793,0.483188,8.105289,2.288054,-1.321756,...,-2.100396,-0.617086,3.363142,-0.316479,-1.854223,7.734058,1.896559,2.982698,-0.530491,4.902455
4,2.0,3.281793,3.414439,0.657914,1.047796,0.238907,1.127898,7.285354,1.667133,-1.228284,...,-2.526155,-0.706397,3.741111,-0.639952,-1.269339,7.916979,1.985572,2.118069,-0.314364,4.778358
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2.0,3.871755,3.589501,0.416867,0.349947,0.337356,1.070687,8.019194,2.064397,-1.305608,...,-2.364719,-1.266991,3.414746,-0.378161,-1.657019,7.796263,2.057745,2.252022,-0.318535,5.028338
76,2.0,3.490173,3.661987,0.347921,0.181220,-0.067744,0.765243,6.982029,2.004046,-1.424740,...,-1.734418,-0.764236,3.619848,-1.299359,-1.275860,7.920062,2.016106,2.539820,-1.281596,4.936462
77,2.0,3.153825,3.404133,0.432284,1.346610,0.237783,0.835472,7.567784,1.901258,-1.012742,...,-2.529978,-1.335582,3.546829,-0.645620,-1.765130,7.951655,2.316642,2.174092,-0.294243,4.908875
78,3.0,4.215745,3.672641,0.926392,1.158929,-0.082361,0.982227,6.831725,1.893408,-1.134000,...,-1.916386,-0.611010,3.474983,-0.549410,-1.476441,7.949315,1.449637,3.179821,-0.635886,4.837736


# Standardize response column naming
We rename the response column to "response" for consistency across model fitting, and we cast the categorical response to integer labels

In [5]:
# Set response column name

source_synth_data_cont.rename(columns={source_synth_data_cont.columns[0]: 'response'}, inplace=True)
target_synth_data_cont.rename(columns={target_synth_data_cont.columns[0]: 'response'}, inplace=True)
display(source_synth_data_cont)
display(target_synth_data_cont)

source_synth_data_cat.rename(columns={source_synth_data_cat.columns[0]: 'response'}, inplace=True)
target_synth_data_cat.rename(columns={target_synth_data_cat.columns[0]: 'response'}, inplace=True)
source_synth_data_cat = source_synth_data_cat.astype({"response": np.int64})
target_synth_data_cat = target_synth_data_cat.astype({"response": np.int64})
display(source_synth_data_cat)
display(target_synth_data_cat)

,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-0.018577,-2.131510,4.221440,7.503035,0.704963,4.994967,1.974920,-1.389276,-2.121761,-2.686822,...,5.012847,1.029205,-1.485498,3.928012,0.794997,0.670302,2.786066,8.491732,-2.031756,0.116399
1,1.647239,-2.541027,4.464055,8.170037,0.448550,5.254451,1.639822,-1.552928,-2.047237,-2.977847,...,5.153622,0.895523,-1.169409,3.986608,0.025046,0.251345,2.805506,9.045304,-2.157403,-0.023953
2,-1.212490,-2.140310,4.584173,8.010408,0.552976,5.319696,2.019203,-1.104839,-3.188980,-2.853624,...,3.255986,0.781753,-1.217508,4.504326,0.987089,0.886421,2.750850,8.694841,-2.431166,-0.693216
3,1.717835,-2.476493,4.213687,7.685149,0.655794,4.797119,2.243716,-1.213976,-0.911602,-3.179669,...,5.890727,0.987917,-1.346762,4.099814,1.204830,0.611966,2.748806,8.128385,-1.751405,0.239025
4,0.828215,-2.559346,4.489350,7.722889,0.816977,4.869013,1.448423,-1.378270,-1.312596,-2.537707,...,6.329980,0.814059,-1.101288,4.431397,0.487904,0.434994,2.410800,8.530311,-1.780194,0.533092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1.132352,-2.575378,4.335910,7.880893,0.511562,4.968101,1.666043,-1.319583,-3.059473,-3.048925,...,4.186401,0.904709,-1.366982,4.172993,1.185925,0.224145,2.943122,8.613989,-2.507009,0.130276
96,0.416058,-2.492329,4.535006,8.181652,0.722679,4.787186,1.217221,-1.232186,-2.207171,-3.422057,...,3.492457,0.878575,-1.523754,4.149942,1.440319,0.675295,2.510847,8.433688,-2.334204,0.418409
97,0.055745,-1.790082,4.298029,7.858824,0.787478,5.324826,2.256585,-1.766166,-0.285702,-3.021066,...,4.941716,0.967434,-1.516619,4.438812,0.084988,0.480843,1.962870,8.318415,-2.064034,0.054013
98,1.435158,-2.712868,3.964359,7.665058,0.461145,5.245588,1.677615,-1.117710,-0.714720,-2.777461,...,3.261225,0.617396,-1.425893,4.535557,0.698371,0.579830,2.726510,8.354184,-2.074454,0.094469


,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,-0.131928,-1.825388,4.387839,8.065345,0.542474,5.000099,1.189423,-1.161513,-0.968273,-2.778407,...,2.322157,0.957780,-1.387260,4.247323,1.862496,0.799520,2.320422,8.470030,-1.849864,0.006803
1,0.145121,-2.554434,4.349408,7.799595,0.387225,5.320900,1.349798,-1.331643,-2.065110,-2.948631,...,3.990292,0.845537,-1.172766,4.262803,1.541075,0.495177,1.948710,8.750320,-1.721811,-0.247589
2,1.622457,-1.943416,4.287696,7.768383,0.370545,5.013423,1.833743,-0.925606,-1.942006,-2.827866,...,3.097417,0.698807,-1.799050,4.567008,0.669068,0.823613,2.333797,8.505746,-2.298207,0.077588
3,1.362953,-1.891210,4.447942,7.572323,0.720992,5.378908,2.000493,-0.968341,-2.112151,-2.904019,...,1.405052,0.890627,-1.695162,3.967584,0.883925,0.932713,2.304135,8.507120,-2.367570,-0.502198
4,1.337630,-2.151685,4.389698,8.134839,0.547072,4.738690,1.571785,-1.034105,-1.566045,-3.522164,...,5.730517,0.954667,-1.422893,4.314065,0.322956,0.751003,2.692535,8.746863,-2.003059,-0.131694
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,-3.238152,-2.887291,4.349457,7.760678,0.592905,4.904736,1.209448,-1.349284,0.546408,-3.096531,...,4.549129,1.065473,-1.395109,4.207340,1.866048,0.395522,2.346771,8.545214,-2.304675,-1.011131
76,1.249571,-2.329088,4.125388,8.215697,0.428899,5.388900,2.020143,-1.191649,-2.372020,-2.690694,...,4.036303,0.840662,-1.407928,3.793202,1.107406,0.622723,2.509919,8.426898,-2.156630,-0.015322
77,0.284994,-2.403273,4.254410,7.832009,0.831893,5.040045,2.132998,-0.949456,-0.619289,-3.058549,...,4.655577,0.843374,-1.447009,4.385909,1.809769,0.916229,2.527795,8.567823,-2.141750,-0.628910
78,-0.294250,-2.271097,4.178750,8.105634,0.627029,5.154989,2.121331,-1.058402,0.449825,-3.090483,...,2.707273,0.884918,-1.787573,4.444146,2.515218,0.306426,2.789173,8.509702,-2.612548,-0.807883


,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,2,3.184250,3.445221,0.772872,1.583466,0.188274,1.153636,7.081006,1.970003,-1.144194,...,-2.264175,-0.338430,3.798686,-0.374314,-0.987292,7.880432,1.669029,2.783176,-0.344493,5.073896
1,1,2.930739,3.444411,0.565156,0.635906,0.603209,0.838928,7.949908,1.815713,-1.043961,...,-1.873429,-0.658501,4.181844,-1.009165,-1.530263,7.770458,1.524477,1.700582,-0.206422,4.744504
2,3,4.061349,3.515256,0.491728,0.941261,0.290420,1.069911,7.539802,2.173966,-1.421109,...,-2.295036,-1.128794,3.475350,-0.672629,-0.816676,8.028649,2.290329,2.503215,-0.620180,4.722170
3,2,3.791040,3.619319,0.509547,0.460347,0.197607,0.954524,7.544457,1.743920,-1.260519,...,-2.001552,-0.127618,3.659228,-0.825454,-1.261197,7.912557,1.713280,2.226306,-0.618549,4.681098
4,1,2.324803,3.404648,0.488172,1.238389,0.618491,0.491636,7.559692,2.095115,-1.114257,...,-2.764871,-0.591379,3.512712,-1.252520,-1.655410,8.000330,2.194835,2.262948,-0.740519,5.113681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,1,3.391414,3.681952,0.414090,1.318297,0.626547,0.641115,7.406402,2.288608,-1.440167,...,-2.065471,-0.570546,3.363025,-0.776700,-0.606547,7.973992,2.158482,2.476712,-0.548647,4.712099
101,3,3.921791,3.649770,0.420173,0.894762,0.247528,0.707007,7.203413,2.131378,-0.779443,...,-2.076712,-0.732730,3.685996,-1.238457,-0.856233,7.811980,1.985250,2.263119,-0.263122,4.949613
102,2,4.233251,3.791196,0.597181,1.173061,-0.034478,0.720862,7.619725,2.062568,-1.043749,...,-2.150644,0.097892,3.276727,-0.888212,-1.241449,7.801340,2.124649,2.132417,0.195482,4.766642
103,2,3.228267,3.563340,0.648522,0.575898,0.073183,1.004983,7.865742,1.855545,-0.940535,...,-2.700999,-0.883108,3.851675,-0.359209,-0.950857,7.855413,2.192433,2.671906,-1.089740,4.788992


,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
0,3,3.587775,3.750322,0.607150,0.011245,0.036504,1.147931,7.326725,1.589885,-1.268487,...,-1.654367,-0.911683,3.811610,-0.667263,-1.172213,7.895552,1.401248,2.664864,-0.891760,5.009576
1,1,2.162534,3.402201,0.582610,1.142536,0.045415,1.054657,6.835663,1.871839,-0.982427,...,-2.118626,-0.711217,3.706629,-1.181281,-1.335658,7.990837,1.993115,2.819070,-0.854748,4.880440
2,1,2.620521,3.395528,0.395628,0.508721,0.264623,0.616177,7.619991,1.785727,-1.117486,...,-2.033089,-0.655606,3.628446,0.146929,-1.233156,7.879871,1.416449,1.545361,0.119349,4.977223
3,2,2.339685,3.789641,0.195729,0.308257,-0.063793,0.483188,8.105289,2.288054,-1.321756,...,-2.100396,-0.617086,3.363142,-0.316479,-1.854223,7.734058,1.896559,2.982698,-0.530491,4.902455
4,2,3.281793,3.414439,0.657914,1.047796,0.238907,1.127898,7.285354,1.667133,-1.228284,...,-2.526155,-0.706397,3.741111,-0.639952,-1.269339,7.916979,1.985572,2.118069,-0.314364,4.778358
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2,3.871755,3.589501,0.416867,0.349947,0.337356,1.070687,8.019194,2.064397,-1.305608,...,-2.364719,-1.266991,3.414746,-0.378161,-1.657019,7.796263,2.057745,2.252022,-0.318535,5.028338
76,2,3.490173,3.661987,0.347921,0.181220,-0.067744,0.765243,6.982029,2.004046,-1.424740,...,-1.734418,-0.764236,3.619848,-1.299359,-1.275860,7.920062,2.016106,2.539820,-1.281596,4.936462
77,2,3.153825,3.404133,0.432284,1.346610,0.237783,0.835472,7.567784,1.901258,-1.012742,...,-2.529978,-1.335582,3.546829,-0.645620,-1.765130,7.951655,2.316642,2.174092,-0.294243,4.908875
78,3,4.215745,3.672641,0.926392,1.158929,-0.082361,0.982227,6.831725,1.893408,-1.134000,...,-1.916386,-0.611010,3.474983,-0.549410,-1.476441,7.949315,1.449637,3.179821,-0.635886,4.837736


# Split target into train, ensemble, and test

* We split the target dataset into:

* train: used for target-side fitting/adaptation

* ensemble: a small held-out target partition used to estimate performance-based weights for combining individual random forest models into a weighted ensemble

* test: final evaluation set

For categorical responses, we stratify splits to ensure each partition contains all classes.

In [6]:
from sklearn.model_selection import train_test_split

target_synth_train_comb_cont, target_synth_test_cont = train_test_split(
    target_synth_data_cont,
    train_size = num_samples_target + samples_target_ensemble,
    test_size = samples_target_test
)

target_synth_train_comb_cat, target_synth_test_cat = train_test_split(
    target_synth_data_cat,
    train_size = num_samples_target + samples_target_ensemble,
    test_size = samples_target_test,
    stratify = target_synth_data_cat['response'] # crucial or else random forest might break due to partition not representing all classes.
)

target_synth_train_cont, target_synth_train_ensemble_cont = train_test_split(
    target_synth_train_comb_cont,
    train_size = num_samples_target,
    test_size = samples_target_ensemble
)

target_synth_train_cat, target_synth_train_ensemble_cat = train_test_split(
    target_synth_train_comb_cat,
    train_size = num_samples_target,
    test_size = samples_target_ensemble,
    stratify = target_synth_train_comb_cat['response'] # crucial or else random forest might break due to partition not representing all classes.
)

# Create DatasetContainer objects
We provide a DatasetContainer helper class to keep the different datasets organized. Optionally, `id_tuple` can also be set which is used to label the scenario and replicate IDs if performing large scale simulation studies. These IDs are passed to the output after model fitting.

In [7]:
from omicstl.simulation_utils.data_utils import DatasetContainer
datasets_cont = DatasetContainer(
	source_data=source_synth_data_cont,
	target_data=target_synth_train_cont,
    target_ensemble_data=target_synth_train_ensemble_cont,
	target_test_data=[target_synth_test_cont]
)
datasets_cont.set_response_column("response") # Identify response column

datasets_cat = DatasetContainer(
	source_data=source_synth_data_cat,
	target_data=target_synth_train_cat,
    target_ensemble_data=target_synth_train_ensemble_cat,
	target_test_data=[target_synth_test_cat]
)
datasets_cat.set_response_column("response") # Identify response column

# Fit transfer-learning models (deep learning and random forest)

We demonstrate both deep learning and random-forest transfer learning models. Deep learning models support automatic tuning over a parameter grid via *fit_dl_model*. Random forest models are fit through a Python interface to the R implementation

In [8]:
param_grid = {
	"dropout": [0.25, 0.5],
	"n_latent_dims": [2],
	"hidden_dim_base": [6],
	"lr": [0.01, 0.001],
	"source_epochs": [1000],
	"target_epochs": [1000],
	"freeze": ["none"],
	"weight_decay": [1e-4, 1e-2],
	"gamma": [1, 2, 3]
}

Deep learning models can be fit using a DatasetContainer and parameter grid using `fit_dl_model`, which returns a dataframe with results, the trained transfer learning model, and the model trained on only the target dataset.

In [9]:
import torch
from torch import device
import random
from omicstl.simulation_utils.model_utils import fit_dl_model

random.seed(42)
torch.manual_seed(42)
out, mult_vae_cont_model, model_targetonly = fit_dl_model(
	datasets_cont,
	"mult_vae",
	device("cpu"),
	param_grid
)
display(out)

random.seed(42)
torch.manual_seed(42)
out, mult_vae_cat_model, model_targetonly = fit_dl_model(
	datasets_cat,
	"mult_vae",
	device("cpu"),
	param_grid
)
display(out)

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_vae,mult_vae,target,1.581632,1.017247,NaN,NaN,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.010,0.0001,1.0
1,None,None,test_0,mult_vae,mult_vae,target_nosource,1.597953,1.060553,NaN,NaN,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.0100,1.0


/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavi

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_vae,mult_vae,target,NaN,NaN,0.56,0.536000,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.0001,2.0
1,None,None,test_0,mult_vae,mult_vae,target_nosource,NaN,NaN,0.36,0.190588,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.0001,1.0


In [10]:
random.seed(42)
torch.manual_seed(42)
out, mult_mlp_cont_model, model_targetonly = fit_dl_model(
	datasets_cont,
	"mult_mlp",
	device("cpu"),
	param_grid
)
display(out)

random.seed(42)
torch.manual_seed(42)
out, mult_mlp_cat_model, model_targetonly = fit_dl_model(
	datasets_cat,
	"mult_mlp",
	device("cpu"),
	param_grid
)
display(out)

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_mlp,mult_mlp,target,1.587929,1.215138,NaN,NaN,...,0.5,6.0,2.0,1000.0,1000.0,none,12.0,0.010,0.0001,1.0
1,None,None,test_0,mult_mlp,mult_mlp,target_nosource,1.717981,1.281345,NaN,NaN,...,0.5,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.0001,2.0


/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1040: FutureWarning: The behavi

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_mlp,mult_mlp,target,NaN,NaN,0.16,0.106667,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.010,0.0001,2.0
1,None,None,test_0,mult_mlp,mult_mlp,target_nosource,NaN,NaN,0.28,0.149333,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.0001,3.0


The random forest based models can also be fit using a python interface to the R modeling code.

In [ ]:
from omicstl.r_utils import set_seed
from omicstl.simulation_utils.model_utils import fit_rf_model

random.seed(42)
out, rf_cont_model = fit_rf_model(datasets_cont)
display(out)

random.seed(42)
set_seed(42)
out, rf_cat_model = fit_rf_model(datasets_cat)
display(out)

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1240: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1240: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,mcc,precision,recall,roc_auc
0,None,None,test_0,rf,pred_source_full,target,1.383936,0.936974,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,test_0,rf,pred_source_full_val,target,1.383936,0.936974,NaN,NaN,NaN,NaN,NaN,NaN
2,None,None,test_0,rf,pred_0_full,target,1.524802,1.081111,NaN,NaN,NaN,NaN,NaN,NaN
3,None,None,test_0,rf,pred_0_full_val,target,1.524802,1.081111,NaN,NaN,NaN,NaN,NaN,NaN
4,None,None,test_0,rf,pred_1_full,target,1.258176,1.002008,NaN,NaN,NaN,NaN,NaN,NaN
5,None,None,test_0,rf,pred_1_full_val,target,1.258176,1.002008,NaN,NaN,NaN,NaN,NaN,NaN
6,None,None,test_0,rf,pred_2_full,target,1.390045,1.014475,NaN,NaN,NaN,NaN,NaN,NaN
7,None,None,test_0,rf,pred_2_full_val,target,1.390045,1.014475,NaN,NaN,NaN,NaN,NaN,NaN
8,None,None,test_0,rf,pred_3_full,target,1.448319,1.047922,NaN,NaN,NaN,NaN,NaN,NaN
9,None,None,test_0,rf,pred_3_full_val,target,1.448319,1.047922,NaN,NaN,NaN,NaN,NaN,NaN


/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1240: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1240: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)
/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1240: F

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,mcc,precision,recall,roc_auc
0,None,None,test_0,rf,pred_source_full,target,NaN,NaN,0.36,0.359481,0.030024,0.377436,0.36,0.682721
1,None,None,test_0,rf,pred_source_full_val,target,NaN,NaN,0.44,0.431429,0.157694,0.490667,0.44,0.682721
2,None,None,test_0,rf,pred_0_full,target,NaN,NaN,0.28,0.251467,-0.099677,0.245905,0.28,0.388088
3,None,None,test_0,rf,pred_0_full_val,target,NaN,NaN,0.40,0.366008,0.093547,0.380952,0.40,0.388088
4,None,None,test_0,rf,pred_1_full,target,NaN,NaN,0.40,0.381697,0.092792,0.448762,0.40,0.676176
5,None,None,test_0,rf,pred_1_full_val,target,NaN,NaN,0.32,0.260000,-0.040362,0.224000,0.32,0.676176
6,None,None,test_0,rf,pred_2_full,target,NaN,NaN,0.44,0.396182,0.159071,0.411810,0.44,0.644412
7,None,None,test_0,rf,pred_2_full_val,target,NaN,NaN,0.32,0.295188,-0.035977,0.300000,0.32,0.644412
8,None,None,test_0,rf,pred_3_full,target,NaN,NaN,0.52,0.500449,0.283695,0.521333,0.52,0.595882
9,None,None,test_0,rf,pred_3_full_val,target,NaN,NaN,0.40,0.379221,0.092574,0.379487,0.40,0.595882


# Predicting data using a pretrained transfer learning model

Once you have created a model, you can then use it to predict response values for novel inputs.
For deep learning models you can use the `predict_dl_model` function.

In [ ]:
from omicstl.simulation_utils.model_utils import predict_dl_model

# Create "new" data by shuffling existing values
new_synth_data_cont = source_synth_data_cont.iloc[:, 1:]
values = new_synth_data_cont.values.flatten()
np.random.shuffle(values)
new_synth_data_cont = pd.DataFrame(
    values.reshape(new_synth_data_cont.shape),
    columns=new_synth_data_cont.columns
)

new_synth_data_cat = source_synth_data_cat.iloc[:, 1:]
values = new_synth_data_cat.values.flatten()
np.random.shuffle(values)
new_synth_data_cat = pd.DataFrame(
    values.reshape(new_synth_data_cat.shape),
    columns=new_synth_data_cat.columns
)

# Get predicted response values on the new data
out = predict_dl_model(mult_vae_cont_model, new_synth_data_cont)
display(out)

out = predict_dl_model(mult_vae_cat_model, new_synth_data_cat)
display(out)

out = predict_dl_model(mult_mlp_cont_model, new_synth_data_cont)
display(out)

out = predict_dl_model(mult_mlp_cat_model, new_synth_data_cat)
display(out)

Similarly, you can use the `predict_rf_model` function to predict response values using a random forest model. Random forest predictions will include the predicted responses of each of the transfer learning methods. You can select the appropriate method by observing the reported accuracies of each method from the original `fit_rf_model` call.

In [ ]:
from omicstl.simulation_utils.model_utils import predict_rf_model

out = predict_rf_model(rf_cont_model, new_synth_data_cont)
display(pd.DataFrame(out))

out = predict_rf_model(rf_cat_model, new_synth_data_cat)
display(pd.DataFrame(out))

# Shapley Values

Users can get Shapley values using the built-in `get_shapley_values_dl` and `get_shapley_values_rf` functions.

In [15]:
from omicstl.shapley import get_shapley_values_dl

source_input_cont = datasets_cont.source_data
target_input_cont = datasets_cont.target_test_data[0]

shap_dl_mlp_cont = get_shapley_values_dl(
    source_input=source_input_cont,
    target_input=target_input_cont,
    pretrained_model=mult_mlp_cont_model,
    background_sample_size=20,
    shap_sample_size=50,
    response_id="response"
)
display(shap_dl_mlp_cont)

shap_dl_vae_cont = get_shapley_values_dl(
    source_input=source_input_cont,
    target_input=target_input_cont,
    pretrained_model=mult_vae_cont_model,
    background_sample_size=20,
    shap_sample_size=50,
    response_id="response"
)
display(shap_dl_vae_cont)

source_input_cat = datasets_cat.source_data
target_input_cat = datasets_cat.target_test_data[0]

shap_dl_cont = get_shapley_values_dl(
    source_input=source_input_cat,
    target_input=target_input_cat,
    pretrained_model=mult_mlp_cat_model,
    background_sample_size=20,
    shap_sample_size=50,
    response_id="response"
)
display(shap_dl_mlp_cont)

shap_dl_cont = get_shapley_values_dl(
    source_input=source_input_cat,
    target_input=target_input_cat,
    pretrained_model=mult_vae_cat_model,
    background_sample_size=20,
    shap_sample_size=50,
    response_id="response"
)
display(shap_dl_vae_cont)


  0%|          | 0/25 [00:00<?, ?it/s]

/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 8 iterations, i.e. alpha=5.531e-03, with an active set of 8 regressors, and the smallest cholesky pivot element being 4.215e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 8 iterations, i.e. alpha=5.531e-03, with an active set of 8 regressors, and the smallest cholesky pivot element being 7.300e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 2 iterations, i.e. alpha=2.075e-02, with an active set of 2 regressors, and the smallest cholesky pivot element being 8.42

,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
9,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.172546,0.000000,0.000000,0.000000,0.0,0.000000
15,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
1,0.0,-0.403390,0.000000,0.000000,-0.044002,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.139301,0.000000,0.0,0.000000
56,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.170934
55,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.091751,...,0.000000,-0.138945,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
23,0.0,0.000000,0.000000,0.000000,-0.053738,0.038153,0.000000,-0.015161,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.150601,0.000000,0.000000,-0.059602,0.0,0.198696
74,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.062249,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
57,0.0,0.000000,0.000000,0.056555,0.055298,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.149969,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
41,0.0,0.079710,0.000000,0.000000,0.124201,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,-0.129275,0.000000,0.000000,0.000000,0.0,0.070712
25,0.0,0.000000,0.000000,0.000000,-0.022738,0.000000,0.000000,0.000000,0.000000,-0.168074,...,0.000000,0.000000,0.000000,0.000000,-0.036551,0.000000,0.000000,0.000000,0.0,0.000000


  0%|          | 0/25 [00:00<?, ?it/s]

,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
55,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
74,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
57,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


...

/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 6 iterations, i.e. alpha=5.090e-03, with an active set of 6 regressors, and the smallest cholesky pivot element being 5.960e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 6 iterations, i.e. alpha=5.090e-03, with an active set of 6 regressors, and the smallest cholesky pivot element being 2.220e-16. Reduce max_iter or increase eps parameters.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 6 iterations, i.e. alpha=5.090e-03, with an active set of 6 regressors, and the smallest cholesky pivot element being 4.21

,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
9,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.172546,0.000000,0.000000,0.000000,0.0,0.000000
15,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
1,0.0,-0.403390,0.000000,0.000000,-0.044002,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.139301,0.000000,0.0,0.000000
56,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.170934
55,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.091751,...,0.000000,-0.138945,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
23,0.0,0.000000,0.000000,0.000000,-0.053738,0.038153,0.000000,-0.015161,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.150601,0.000000,0.000000,-0.059602,0.0,0.198696
74,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.062249,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
57,0.0,0.000000,0.000000,0.056555,0.055298,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.149969,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
41,0.0,0.079710,0.000000,0.000000,0.124201,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,-0.129275,0.000000,0.000000,0.000000,0.0,0.070712
25,0.0,0.000000,0.000000,0.000000,-0.022738,0.000000,0.000000,0.000000,0.000000,-0.168074,...,0.000000,0.000000,0.000000,0.000000,-0.036551,0.000000,0.000000,0.000000,0.0,0.000000


  0%|          | 0/25 [00:00<?, ?it/s]

/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 6 iterations, i.e. alpha=1.004e-02, with an active set of 6 regressors, and the smallest cholesky pivot element being 8.429e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 7 iterations, i.e. alpha=8.204e-03, with an active set of 7 regressors, and the smallest cholesky pivot element being 8.429e-08. Reduce max_iter or increase eps parameters.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 8 iterations, i.e. alpha=1.228e-02, with an active set of 8 regressors, and the smallest cholesky pivot element being 2.22

,response,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,100
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
55,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
74,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
57,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
from omicstl.shapley import get_shapley_values_rf

shap_dl_cont = get_shapley_values_rf(
    source_input=source_input_cont,
    target_input=target_input_cont,
    pretrained_model=rf_cont_model,
    background_sample_size=20,
    shap_sample_size=50,
    response_id="response"
)
display(shap_dl_cont)

shap_dl_cont = get_shapley_values_rf(
    source_input=source_input_cont,
    target_input=target_input_cont,
    pretrained_model=rf_cat_model,
    background_sample_size=20,
    shap_sample_size=50,
    response_id="response"
)
display(shap_dl_cont)

ValueError: cannot call `vectorize` on size 0 inputs unless `otypes` is set

# Advanced usage
Advanced users can also work directly with the base classes we provide for each TL model via TransferForest() and MultiViewModel(). 